# Structured Output & Constrained Decoding

## Why this matters

Every LLM call that needs to feed a downstream system (a JSON parser, a database, another function) has the same fundamental problem: the model generates text token-by-token, and free text has no guarantee of matching a schema. This notebook covers the three approaches to solving that, why two of them are fragile, and why the third (what we now use in the X++ review agent's `base_agent.py`) is structurally different, not just "more careful prompting."

## Level 1: What it is

Three approaches exist, in order of increasing reliability:

1. **Prompt-and-hope** — ask the model to "return JSON", parse the text response with `json.loads()`. This is what the X++ project's `base_agent.py` did originally.
2. **Prompt-and-repair** — same as above, but with a defensive parsing layer (regex extraction, bracket matching, control-character escaping) to catch the common ways free text breaks. This is what `base_agent.py` evolved into before the tool-use rewrite.
3. **Constrained decoding / tool-use** — the API itself restricts which tokens the model can generate at each step, so the output is *guaranteed* to match a schema before it ever reaches your code. This is what Claude's tool-use API does, and what OpenAI calls "function calling" or "structured outputs" (`json_schema` mode).

## Level 2: How it actually works internally

### Prompt-and-hope / prompt-and-repair (approaches 1–2)

The model is doing next-token prediction with no structural constraint. "Return valid JSON" is *advisory*, not *enforced* — it's a strong prior from training (the model has seen millions of JSON examples and is good at producing them), but nothing in the decoding process prevents it from emitting a stray unescaped newline inside a string value, or getting cut off mid-object if it hits `max_tokens` before closing every brace. The model doesn't "know" it's mid-JSON in a structural sense — it's predicting the statistically likely next token given everything so far, and 99% of the time that's schema-valid, which is exactly what makes the 1% failure mode so hard to catch in testing.

### Constrained decoding / tool-use (approach 3)

This works differently at the decoding level. When you pass a JSON Schema via `tools=[...]` and force it with `tool_choice`, the serving infrastructure uses the schema to build a **grammar** — a formal description of every valid next-token sequence at each position in the output. During generation, instead of sampling freely from the full vocabulary at each step, the decoder's logits are masked so that only tokens which keep the output on a path toward a schema-valid completion are eligible to be selected.

Concretely: if the schema says the next field must be an enum with values `["Critical","Major","Minor","Info"]`, the model literally cannot emit a token sequence that isn't one of those four strings at that position — it's not "very likely" to comply, it's *structurally incapable* of not complying, the same way a regex engine can't match text outside its pattern.

This is why replacing `base_agent.py`'s `json.loads()`-and-repair logic with tool-use didn't just make failures less likely — it removed an entire class of failure. There's no longer a "the model emitted malformed JSON" case to defend against, because malformed JSON isn't in the space of things the decoder can produce when `tool_choice` forces a schema.

## Level 3: Where it breaks, and what it doesn't solve

Constrained decoding fixes **syntactic** validity, not **semantic** correctness. Know the difference — this is the #1 thing people get wrong when they think tool-use "solves" structured output:

1. **Truncation is still possible.** A schema-valid object can still get cut off mid-generation if `max_tokens` is hit before the model finishes — the grammar constrains *which* tokens are legal, not *how many* tokens are available. This is why `base_agent.py` still checks `response.stop_reason == "max_tokens"` and raises explicitly, even after moving to tool-use.

2. **The model can still be *wrong* within a valid schema.** Forcing `severity` to be one of four enum values doesn't stop the model from picking "Critical" when "Major" would have been the better judgment. This is exactly the LLM-judgment variance still seen in the Security/Performance agents' non-deterministic findings — schema enforcement and judgment quality are completely orthogonal.

3. **Grammar construction has a performance cost.** Very large or deeply nested schemas can slow generation, since the decoder has to compute the valid-token mask at every step. Not a concern for a schema the size of our issue schema, but matters if you're ever tempted to build an enormous nested schema for a huge structured extraction task.

4. **It doesn't validate cross-field logic.** JSON Schema can say "severity must be one of these 4 strings" but can't easily express "if severity is Critical, consequence must be non-empty" — conditional/cross-field constraints need either a more advanced schema (JSON Schema does support some conditionals via if/then, but it gets unwieldy fast) or a post-hoc validation layer.

5. **Not every provider implements this the same way.** OpenAI's "structured outputs" mode (as opposed to older function calling) guarantees schema compliance via a similar grammar-constraint approach. Older function-calling implementations across providers were closer to "strongly encouraged" than "enforced" — worth verifying per-provider/per-mode rather than assuming universal guarantees, since this space moved fast and older docs may describe the weaker behavior.

In [ ]:
# Minimal example: the schema that forces severity into a closed set,
# same pattern as base_agent.py's ISSUE_TOOL

from anthropic import Anthropic

client = Anthropic()

ISSUE_TOOL = {
    "name": "report_issue",
    "description": "Report a single code review issue.",
    "input_schema": {
        "type": "object",
        "properties": {
            "severity": {
                "type": "string",
                "enum": ["Critical", "Major", "Minor", "Info"],
            },
            "title": {"type": "string"},
        },
        "required": ["severity", "title"],
    },
}

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=500,
    tools=[ISSUE_TOOL],
    tool_choice={"type": "tool", "name": "report_issue"},
    messages=[{"role": "user", "content": "A method has an empty catch block."}],
)

tool_use_block = next(b for b in response.content if b.type == "tool_use")
print(tool_use_block.input)
# {'severity': 'Critical', 'title': 'Empty catch block swallows exceptions'}
# severity is GUARANTEED to be one of the 4 enum values -- try passing
# a prompt engineered to make the model say "Extreme" and watch it still
# resolve to one of the 4 allowed strings.

## Interview-ready summary

**Q: "How do you ensure an LLM's output is valid JSON?"**

Weak answer: "I tell it to return JSON and parse the response."

Strong answer: "Free-text-then-parse is inherently probabilistic — the model is very likely but not guaranteed to produce valid JSON, and failure modes like truncation or unescaped control characters inside string values are common enough to matter in production. I use constrained decoding via tool-use / function-calling with an enforced JSON Schema, which restricts the token-generation process itself so the output is structurally guaranteed to match the schema — it's not 'the model is good at JSON,' it's 'the decoder literally cannot emit tokens outside the schema.' I still handle truncation via `stop_reason`, since schema-guarantee only covers well-formedness, not completeness — and I still treat the *values themselves* as needing separate quality review, since a syntactically valid enum pick can still be the wrong judgment call."